# 04 — Feature Engineering

**Pipeline role.** Fourth notebook. Turns the cleaned training data into a justified, interpretable final feature set that downstream models will consume.

**Rubric coverage (from `Group Project Guideline.md`).**
- Report Section 3 — Model Building (feature construction / selection subsection): "feature normalization, feature discretization, feature selection" effort is documented here.
- Provides the **feature decision log** that the report uses to justify modelling choices.

**TL;DR for teammates.** All engineered features must be created on training data only (fit-on-train / apply-on-test). Every kept feature needs a one-line rationale. Every dropped feature needs a one-line reason. No fabricated interactions just to fill space.


## Inputs, Outputs, Artifacts

**Inputs.**
- `data/processed/X_train_v1.*`, `X_test_v1.*`, `y_train_v1.*`, `y_test_v1.*` from notebook 03.
- Preprocessing policy notes from notebook 03.
- Observations from notebook 02 (distributions, skewness, cardinality).

**Outputs.**
- `data/processed/X_train_feat_v1.*`, `X_test_feat_v1.*` — feature-engineered matrices, same row order as inputs.
- In-notebook **feature decision log**: table of `feature_name | source_columns | transform | rationale | keep?`.
- `reports/tables/feature_set_v1.csv` — final kept-feature list and dtype.
- Feature-set version label (`feature_set_v1`) to reference in the experiment log.

**Downstream consumers.**
- Notebook 05 uses `feature_set_v1` for all baselines.
- Notebook 06 may create `feature_set_v2` if tuning reveals benefit; changes must be logged.
- Notebook 08 imports the feature decision log into the report.


## Methodological Influences

Feature-engineering decisions follow standard ISOM3360 practice: derive features inside the training split, keep transformations interpretable, and record every keep/drop decision. See the shared [project conventions](../references/project_conventions.md) for standards applied across all notebooks.


## Key Questions To Answer Here

1. Which derived features are plausibly predictive of cancellation, and why?
2. Which transformations (e.g., `log1p`) are appropriate for scale-sensitive models?
3. How are high-cardinality categoricals (`country`, `agent`, `company`) handled?
4. Which features are excluded, and for what reason?
5. What is the final feature matrix shape and dtype profile?
6. Does the feature set avoid any post-outcome information (no leakage)?


## Work Plan

### 4.1 Setup
- Load processed train/test splits from notebook 03.
- Re-assert shapes and target proportions.

### 4.2 Candidate Derived Features
Classical, interpretable derivations — prefer these before any complex transform:

- `total_nights` = `stays_in_weekend_nights` + `stays_in_week_nights`.
- `total_guests` = `adults` + `children` + `babies` (fill missing children with 0 per NB03 policy).
- `is_family` = indicator (`children > 0` or `babies > 0`).
- `has_special_request` = indicator (`total_of_special_requests > 0`).
- `has_previous_cancellation` = indicator (`previous_cancellations > 0`).
- `adr_per_person` = `adr / max(total_guests, 1)`.
- `room_mismatch` = indicator (`reserved_room_type != assigned_room_type`) — only if NB02 confirms `assigned_room_type` is safe; otherwise drop.
- Calendar features from `arrival_date_*`: `arrival_season`, `is_summer`, `is_weekend_arrival` (derived from year+week+day).

### 4.3 Transformations
- Skewed numerics (e.g., `lead_time`, `adr`): consider `log1p` for linear models; trees can use raw.
- Keep a variant with and without transformation when it meaningfully affects linear models.

### 4.4 High-Cardinality Encoding
- `country`: group rare levels into `"Other"` at a chosen threshold (e.g., <0.5% frequency), then one-hot; rather than target encoding, which would require nested CV to remain leak-free.
- `agent`, `company`: binary "has_agent" / "has_company" flags; keep top-k IDs if predictive.

### 4.5 Feature Selection (Optional, Documented If Used)
- Variance filter, correlation filter, model-based importance (e.g., `SelectFromModel` with L1 logistic regression or a tree ensemble).
- Document which selector was used and why, and keep the pre-selection feature list for reference.

### 4.6 Feature Decision Log
Maintain a table with one row per feature kept or dropped:

| feature_name | source_columns | transform | rationale | keep? |
|---|---|---|---|---|

### 4.7 Save Artifacts
- Save feature-engineered matrices to `data/processed/` with `feature_set_v1` label.
- Save the decision log to `reports/tables/feature_set_v1.csv`.


## Implementation Block 4.1 — Setup

**Scope.** Section 4.1 only — load the frozen train/test artifacts produced by Notebook 03, confirm that the files are present, and re-assert the basic shape and class-balance assumptions before any feature engineering is performed.

No new features are created in this step. The purpose is to establish a stable, versioned starting point for all later feature work in this notebook.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ARTIFACT_VERSION = "v1"
TARGET_COLUMN = "is_canceled"

PROCESSED_DATA_CANDIDATES = [
    Path("../data/processed"),
    Path("data/processed"),
]
PROCESSED_DATA_DIR = next(
    (path for path in PROCESSED_DATA_CANDIDATES if path.exists()),
    PROCESSED_DATA_CANDIDATES[0],
)

if not PROCESSED_DATA_DIR.exists():
    raise FileNotFoundError("The processed data directory from Notebook 03 was not found.")

X_train_path = PROCESSED_DATA_DIR / f"X_train_{ARTIFACT_VERSION}.csv"
X_test_path = PROCESSED_DATA_DIR / f"X_test_{ARTIFACT_VERSION}.csv"
y_train_path = PROCESSED_DATA_DIR / f"y_train_{ARTIFACT_VERSION}.csv"
y_test_path = PROCESSED_DATA_DIR / f"y_test_{ARTIFACT_VERSION}.csv"

required_paths = [X_train_path, X_test_path, y_train_path, y_test_path]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing expected Notebook 03 artifacts: {missing_paths}")

X_train = pd.read_csv(X_train_path)
X_test = pd.read_csv(X_test_path)
y_train = pd.read_csv(y_train_path)[TARGET_COLUMN]
y_test = pd.read_csv(y_test_path)[TARGET_COLUMN]

if len(X_train) != len(y_train):
    raise ValueError("X_train and y_train row counts do not match.")
if len(X_test) != len(y_test):
    raise ValueError("X_test and y_test row counts do not match.")

feature_setup_summary = pd.DataFrame(
    {
        "check": [
            "processed_data_dir",
            "X_train_shape",
            "X_test_shape",
            "y_train_rows",
            "y_test_rows",
            "train_cancellation_rate_pct",
            "test_cancellation_rate_pct",
        ],
        "value": [
            str(PROCESSED_DATA_DIR.resolve()),
            str(X_train.shape),
            str(X_test.shape),
            len(y_train),
            len(y_test),
            round(y_train.mean() * 100, 2),
            round(y_test.mean() * 100, 2),
        ],
    }
)

display(feature_setup_summary)
display(X_train.head())


,check,value
0,processed_data_dir,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
1,X_train_shape,"(69577, 29)"
2,X_test_shape,"(17395, 29)"
3,y_train_rows,69577
4,y_test_rows,17395
5,train_cancellation_rate_pct,27.31
6,test_cancellation_rate_pct,27.31


,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,City Hotel,8,2016,October,41,6,0,1,1,0.0,...,A,0,No Deposit,NaN,40.0,0,Transient,65.00,0,1
1,Resort Hotel,208,2017,April,16,22,2,5,2,0.0,...,E,0,No Deposit,250.0,NaN,0,Transient,71.28,0,0
2,Resort Hotel,0,2017,March,10,11,1,1,1,0.0,...,D,0,No Deposit,241.0,NaN,0,Transient,64.00,0,1
3,City Hotel,102,2017,June,26,30,1,2,2,0.0,...,A,0,No Deposit,9.0,NaN,0,Transient-Party,135.00,1,0
4,City Hotel,3,2016,July,29,11,1,0,1,0.0,...,A,0,No Deposit,NaN,40.0,0,Transient,65.00,0,0


## Implementation Block 4.2 — Candidate Derived Features

**Scope.** Section 4.2 only — create a first set of interpretable, low-risk derived features on both the training and test splits using deterministic row-wise transformations.

Only features that are clearly available at booking time and do not require fitting are created here. `room_mismatch` is deliberately deferred because `assigned_room_type` is still under timing review from Notebook 03.

The purpose of this block is to establish a justified starting feature set before any transformation or encoding decisions are extended further.


In [2]:
month_to_season = {
    "December": "winter",
    "January": "winter",
    "February": "winter",
    "March": "spring",
    "April": "spring",
    "May": "spring",
    "June": "summer",
    "July": "summer",
    "August": "summer",
    "September": "autumn",
    "October": "autumn",
    "November": "autumn",
}

X_train_feat = X_train.copy()
X_test_feat = X_test.copy()

for feature_frame in [X_train_feat, X_test_feat]:
    children_filled = feature_frame["children"].fillna(0)
    total_guests = feature_frame["adults"] + children_filled + feature_frame["babies"]

    feature_frame["total_nights"] = (
        feature_frame["stays_in_weekend_nights"] + feature_frame["stays_in_week_nights"]
    )
    feature_frame["total_guests"] = total_guests
    feature_frame["is_family"] = (
        (children_filled > 0) | (feature_frame["babies"] > 0)
    ).astype(int)
    feature_frame["has_special_request"] = (
        feature_frame["total_of_special_requests"] > 0
    ).astype(int)
    feature_frame["has_previous_cancellation"] = (
        feature_frame["previous_cancellations"] > 0
    ).astype(int)
    feature_frame["adr_per_person"] = feature_frame["adr"] / total_guests.clip(lower=1)
    feature_frame["arrival_season"] = feature_frame["arrival_date_month"].map(month_to_season)
    feature_frame["is_summer"] = feature_frame["arrival_date_month"].isin(
        ["June", "July", "August"]
    ).astype(int)

candidate_feature_summary = pd.DataFrame(
    [
        {
            "feature_name": "total_nights",
            "source_columns": "stays_in_weekend_nights, stays_in_week_nights",
            "transform": "sum",
            "rationale": "Total length of stay may affect cancellation behavior.",
            "status": "created",
        },
        {
            "feature_name": "total_guests",
            "source_columns": "adults, children, babies",
            "transform": "sum with children missing filled as 0",
            "rationale": "Party size can influence booking commitment and rate sensitivity.",
            "status": "created",
        },
        {
            "feature_name": "is_family",
            "source_columns": "children, babies",
            "transform": "binary indicator",
            "rationale": "Family travel may show different cancellation patterns from solo or couple travel.",
            "status": "created",
        },
        {
            "feature_name": "has_special_request",
            "source_columns": "total_of_special_requests",
            "transform": "binary indicator",
            "rationale": "Requests may proxy booking intent and engagement.",
            "status": "created",
        },
        {
            "feature_name": "has_previous_cancellation",
            "source_columns": "previous_cancellations",
            "transform": "binary indicator",
            "rationale": "Past cancellation behavior may signal future cancellation risk.",
            "status": "created",
        },
        {
            "feature_name": "adr_per_person",
            "source_columns": "adr, total_guests",
            "transform": "ratio",
            "rationale": "Per-person room cost may be more interpretable than raw ADR alone.",
            "status": "created",
        },
        {
            "feature_name": "arrival_season",
            "source_columns": "arrival_date_month",
            "transform": "month-to-season mapping",
            "rationale": "Seasonality may capture travel demand patterns more compactly than month names alone.",
            "status": "created",
        },
        {
            "feature_name": "is_summer",
            "source_columns": "arrival_date_month",
            "transform": "binary indicator",
            "rationale": "Summer travel may reflect a distinct demand regime.",
            "status": "created",
        },
        {
            "feature_name": "room_mismatch",
            "source_columns": "reserved_room_type, assigned_room_type",
            "transform": "binary indicator",
            "rationale": "Potentially useful, but still deferred pending timing-safety review of assigned_room_type.",
            "status": "deferred",
        },
    ]
)

feature_shape_summary = pd.DataFrame(
    {
        "dataset": ["X_train_before", "X_train_after", "X_test_before", "X_test_after"],
        "rows": [X_train.shape[0], X_train_feat.shape[0], X_test.shape[0], X_test_feat.shape[0]],
        "columns": [X_train.shape[1], X_train_feat.shape[1], X_test.shape[1], X_test_feat.shape[1]],
    }
)

display(candidate_feature_summary)
display(feature_shape_summary)
display(X_train_feat.head())


,feature_name,source_columns,transform,rationale,status
0,total_nights,"stays_in_weekend_nights, stays_in_week_nights",sum,Total length of stay may affect cancellation b...,created
1,total_guests,"adults, children, babies",sum with children missing filled as 0,Party size can influence booking commitment an...,created
2,is_family,"children, babies",binary indicator,Family travel may show different cancellation ...,created
3,has_special_request,total_of_special_requests,binary indicator,Requests may proxy booking intent and engagement.,created
4,has_previous_cancellation,previous_cancellations,binary indicator,Past cancellation behavior may signal future c...,created
5,adr_per_person,"adr, total_guests",ratio,Per-person room cost may be more interpretable...,created
6,arrival_season,arrival_date_month,month-to-season mapping,Seasonality may capture travel demand patterns...,created
7,is_summer,arrival_date_month,binary indicator,Summer travel may reflect a distinct demand re...,created
8,room_mismatch,"reserved_room_type, assigned_room_type",binary indicator,"Potentially useful, but still deferred pending...",deferred


,dataset,rows,columns
0,X_train_before,69577,29
1,X_train_after,69577,37
2,X_test_before,17395,29
3,X_test_after,17395,37


,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,required_car_parking_spaces,total_of_special_requests,total_nights,total_guests,is_family,has_special_request,has_previous_cancellation,adr_per_person,arrival_season,is_summer
0,City Hotel,8,2016,October,41,6,0,1,1,0.0,...,0,1,1,1.0,0,1,0,65.00,autumn,0
1,Resort Hotel,208,2017,April,16,22,2,5,2,0.0,...,0,0,7,2.0,0,0,0,35.64,spring,0
2,Resort Hotel,0,2017,March,10,11,1,1,1,0.0,...,0,1,2,1.0,0,1,0,64.00,spring,0
3,City Hotel,102,2017,June,26,30,1,2,2,0.0,...,1,0,3,2.0,0,0,0,67.50,summer,1
4,City Hotel,3,2016,July,29,11,1,0,1,0.0,...,0,0,1,1.0,0,0,0,65.00,summer,1


## Implementation Block 4.3 — Transformations

**Scope.** Section 4.3 only — create transformed numeric variants for features that are plausibly right-skewed and may benefit scale-sensitive linear models.

The raw variables are retained. This step adds optional transformed views rather than replacing the original features, which keeps the feature set interpretable and preserves flexibility for both linear and tree-based models.

The purpose of this block is to prepare a small, justified set of transformation candidates before any high-cardinality encoding decisions are applied.


In [3]:
import numpy as np

transformation_candidates = [
    ("lead_time", "log_lead_time"),
    ("adr", "log_adr"),
    ("adr_per_person", "log_adr_per_person"),
    ("total_nights", "log_total_nights"),
]

for source_column, transformed_column in transformation_candidates:
    X_train_feat[transformed_column] = np.log1p(X_train_feat[source_column].clip(lower=0))
    X_test_feat[transformed_column] = np.log1p(X_test_feat[source_column].clip(lower=0))

transformation_summary = pd.DataFrame(
    [
        {
            "feature_name": transformed_column,
            "source_column": source_column,
            "transform": "log1p after non-negative clipping",
            "rationale": "Creates a compressed variant for potentially right-skewed numeric behavior in linear models.",
            "raw_feature_retained": True,
        }
        for source_column, transformed_column in transformation_candidates
    ]
)

transformation_shape_summary = pd.DataFrame(
    {
        "dataset": ["X_train_after_transform", "X_test_after_transform"],
        "rows": [X_train_feat.shape[0], X_test_feat.shape[0]],
        "columns": [X_train_feat.shape[1], X_test_feat.shape[1]],
    }
)

display(transformation_summary)
display(transformation_shape_summary)
display(
    X_train_feat[
        [source for source, _ in transformation_candidates]
        + [target for _, target in transformation_candidates]
    ].head()
)


,feature_name,source_column,transform,rationale,raw_feature_retained
0,log_lead_time,lead_time,log1p after non-negative clipping,Creates a compressed variant for potentially r...,True
1,log_adr,adr,log1p after non-negative clipping,Creates a compressed variant for potentially r...,True
2,log_adr_per_person,adr_per_person,log1p after non-negative clipping,Creates a compressed variant for potentially r...,True
3,log_total_nights,total_nights,log1p after non-negative clipping,Creates a compressed variant for potentially r...,True


,dataset,rows,columns
0,X_train_after_transform,69577,41
1,X_test_after_transform,17395,41


,lead_time,adr,adr_per_person,total_nights,log_lead_time,log_adr,log_adr_per_person,log_total_nights
0,8,65.00,65.00,1,2.197225,4.189655,4.189655,0.693147
1,208,71.28,35.64,7,5.342334,4.280547,3.601141,2.079442
2,0,64.00,64.00,2,0.000000,4.174387,4.174387,1.098612
3,102,135.00,67.50,3,4.634729,4.912655,4.226834,1.386294
4,3,65.00,65.00,1,1.386294,4.189655,4.189655,0.693147


## Implementation Block 4.4 — High-Cardinality Handling

**Scope.** Section 4.4 only — create simple, train-aware handling for the high-cardinality fields without committing to a full target-encoding or top-k ID strategy.

The current policy is conservative. `country` is grouped using frequency thresholds learned from the training split only, while `agent` and `company` contribute booking-presence flags rather than raw identifier expansion. The original columns are retained for now so later feature-selection decisions remain reviewable.

The purpose of this block is to create leakage-safe, interpretable high-cardinality feature variants before the feature decision log and artifact save steps are finalized.


In [4]:
country_frequency_threshold = 0.005

train_country_frequency = (
    X_train_feat["country"]
    .fillna("Unknown")
    .value_counts(normalize=True)
)
frequent_countries = set(
    train_country_frequency[train_country_frequency >= country_frequency_threshold].index
)

for feature_frame in [X_train_feat, X_test_feat]:
    feature_frame["country_grouped"] = (
        feature_frame["country"]
        .fillna("Unknown")
        .where(feature_frame["country"].fillna("Unknown").isin(frequent_countries), "Other")
    )
    feature_frame["has_agent"] = feature_frame["agent"].fillna(0).gt(0).astype(int)
    feature_frame["has_company"] = feature_frame["company"].fillna(0).gt(0).astype(int)

high_cardinality_feature_summary = pd.DataFrame(
    [
        {
            "feature_name": "country_grouped",
            "source_columns": "country",
            "transform": f"group rare levels below {country_frequency_threshold:.3f} training frequency into 'Other'",
            "rationale": "Preserves country signal while limiting sparse categories and ensuring the rule is learned on train only.",
            "status": "created",
        },
        {
            "feature_name": "has_agent",
            "source_columns": "agent",
            "transform": "binary indicator for non-missing positive agent ID",
            "rationale": "Presence of an intermediary may be more stable than the raw high-cardinality identifier.",
            "status": "created",
        },
        {
            "feature_name": "has_company",
            "source_columns": "company",
            "transform": "binary indicator for non-missing positive company ID",
            "rationale": "Presence of a company booking channel may matter even if raw company IDs are too sparse.",
            "status": "created",
        },
        {
            "feature_name": "top_k_agent_or_company_ids",
            "source_columns": "agent, company",
            "transform": "deferred",
            "rationale": "Raw ID expansion is postponed until model evidence justifies the additional complexity.",
            "status": "deferred",
        },
    ]
)

high_cardinality_scope_summary = pd.DataFrame(
    {
        "check": [
            "frequent_countries_retained",
            "country_frequency_threshold",
            "X_train_columns_after_high_cardinality_step",
            "X_test_columns_after_high_cardinality_step",
        ],
        "value": [
            len(frequent_countries),
            country_frequency_threshold,
            X_train_feat.shape[1],
            X_test_feat.shape[1],
        ],
    }
)

display(high_cardinality_feature_summary)
display(high_cardinality_scope_summary)
display(X_train_feat[["country", "country_grouped", "agent", "has_agent", "company", "has_company"]].head())


,feature_name,source_columns,transform,rationale,status
0,country_grouped,country,group rare levels below 0.005 training frequen...,Preserves country signal while limiting sparse...,created
1,has_agent,agent,binary indicator for non-missing positive agen...,Presence of an intermediary may be more stable...,created
2,has_company,company,binary indicator for non-missing positive comp...,Presence of a company booking channel may matt...,created
3,top_k_agent_or_company_ids,"agent, company",deferred,Raw ID expansion is postponed until model evid...,deferred


,check,value
0,frequent_countries_retained,21.000
1,country_frequency_threshold,0.005
2,X_train_columns_after_high_cardinality_step,44.000
3,X_test_columns_after_high_cardinality_step,44.000


,country,country_grouped,agent,has_agent,company,has_company
0,PRT,PRT,NaN,0,40.0,1
1,IRL,IRL,250.0,1,NaN,0
2,DNK,Other,241.0,1,NaN,0
3,BEL,BEL,9.0,1,NaN,0
4,PRT,PRT,NaN,0,40.0,1


## Implementation Block 4.5 — Feature Selection Decision

**Scope.** Section 4.5 only — record whether an automated feature-selection procedure is applied in `feature_set_v1`, and if not, explain that decision explicitly.

For the current version, no automated selector is applied. The feature set remains fully reviewable because the notebook is still in the first engineering pass, and all derived features are interpretable and low in count.

The purpose of this block is to document that feature selection was considered, but deferred until baseline model evidence justifies pruning.


In [5]:
current_feature_columns = X_train_feat.columns.tolist()

feature_selection_decision_summary = pd.DataFrame(
    [
        {
            "decision_area": "automated_selector_used",
            "value": "no",
            "rationale": "The first feature-engineering pass remains small enough to review directly without variance, correlation, or model-based pruning.",
        },
        {
            "decision_area": "selector_deferred_to",
            "value": "Notebook 06 if baseline evidence warrants it",
            "rationale": "Selection should be driven by model evidence rather than applied prematurely during the initial feature-set definition.",
        },
        {
            "decision_area": "current_feature_count",
            "value": len(current_feature_columns),
            "rationale": "The current feature count is still manageable for baseline modeling and interpretation.",
        },
        {
            "decision_area": "timing_review_features_still_deferred",
            "value": "assigned_room_type-derived room_mismatch, raw top-k agent/company IDs",
            "rationale": "These candidates remain unresolved for timing-safety or complexity reasons and are excluded from feature_set_v1 for now.",
        },
    ]
)

feature_selection_scope_summary = pd.DataFrame(
    {
        "dataset": ["X_train_current", "X_test_current"],
        "rows": [X_train_feat.shape[0], X_test_feat.shape[0]],
        "columns": [X_train_feat.shape[1], X_test_feat.shape[1]],
    }
)

display(feature_selection_decision_summary)
display(feature_selection_scope_summary)


,decision_area,value,rationale
0,automated_selector_used,no,The first feature-engineering pass remains sma...
1,selector_deferred_to,Notebook 06 if baseline evidence warrants it,Selection should be driven by model evidence r...
2,current_feature_count,44,The current feature count is still manageable ...
3,timing_review_features_still_deferred,"assigned_room_type-derived room_mismatch, raw ...",These candidates remain unresolved for timing-...


,dataset,rows,columns
0,X_train_current,69577,44
1,X_test_current,17395,44


## Implementation Block 4.6 — Feature Decision Log

**Scope.** Section 4.6 only — consolidate the engineered-feature decisions into a single table that records source columns, transform type, rationale, and whether the feature is currently kept in `feature_set_v1`.

The log focuses on the features actively created or explicitly deferred in this notebook. It is intended to be report-ready and to support the artifact save step that follows.


In [6]:
feature_decision_log = pd.concat(
    [
        candidate_feature_summary.rename(columns={"status": "decision"}),
        transformation_summary.rename(
            columns={
                "source_column": "source_columns",
                "raw_feature_retained": "decision",
            }
        ).assign(
            decision=lambda table: table["decision"].map({True: "keep", False: "drop"})
        ),
        high_cardinality_feature_summary.rename(columns={"status": "decision"}),
    ],
    ignore_index=True,
    sort=False,
)

feature_decision_log["decision"] = feature_decision_log["decision"].replace(
    {"created": "keep", "deferred": "defer"}
)
feature_decision_log = feature_decision_log[
    ["feature_name", "source_columns", "transform", "rationale", "decision"]
]

feature_decision_log_summary = pd.DataFrame(
    {
        "check": [
            "decision_log_rows",
            "kept_features_logged",
            "deferred_features_logged",
        ],
        "value": [
            len(feature_decision_log),
            int((feature_decision_log["decision"] == "keep").sum()),
            int((feature_decision_log["decision"] == "defer").sum()),
        ],
    }
)

display(feature_decision_log)
display(feature_decision_log_summary)


,feature_name,source_columns,transform,rationale,decision
0,total_nights,"stays_in_weekend_nights, stays_in_week_nights",sum,Total length of stay may affect cancellation b...,keep
1,total_guests,"adults, children, babies",sum with children missing filled as 0,Party size can influence booking commitment an...,keep
2,is_family,"children, babies",binary indicator,Family travel may show different cancellation ...,keep
3,has_special_request,total_of_special_requests,binary indicator,Requests may proxy booking intent and engagement.,keep
4,has_previous_cancellation,previous_cancellations,binary indicator,Past cancellation behavior may signal future c...,keep
5,adr_per_person,"adr, total_guests",ratio,Per-person room cost may be more interpretable...,keep
6,arrival_season,arrival_date_month,month-to-season mapping,Seasonality may capture travel demand patterns...,keep
7,is_summer,arrival_date_month,binary indicator,Summer travel may reflect a distinct demand re...,keep
8,room_mismatch,"reserved_room_type, assigned_room_type",binary indicator,"Potentially useful, but still deferred pending...",defer
9,log_lead_time,lead_time,log1p after non-negative clipping,Creates a compressed variant for potentially r...,keep


,check,value
0,decision_log_rows,17
1,kept_features_logged,15
2,deferred_features_logged,2


## Implementation Block 4.7 — Save Artifacts

**Scope.** Section 4.7 only — freeze the current engineered train/test matrices as `feature_set_v1`, save the feature decision log for reporting, and record the resulting artifact paths.

This step does not alter the feature definitions. It only persists the current `feature_set_v1` outputs so downstream modeling notebooks can consume a stable, versioned feature set.


In [7]:
FEATURE_SET_VERSION = "feature_set_v1"

processed_dir = PROCESSED_DATA_DIR
report_tables_dir = Path("../reports/tables")
if not report_tables_dir.exists():
    report_tables_dir = Path("reports/tables")
report_tables_dir.mkdir(parents=True, exist_ok=True)

X_train_feat_path = processed_dir / "X_train_feat_v1.csv"
X_test_feat_path = processed_dir / "X_test_feat_v1.csv"
feature_log_path = report_tables_dir / "feature_set_v1.csv"

X_train_feat.to_csv(X_train_feat_path, index=False)
X_test_feat.to_csv(X_test_feat_path, index=False)
feature_decision_log.to_csv(feature_log_path, index=False)

feature_artifact_summary = pd.DataFrame(
    {
        "artifact": [
            "X_train_feat_v1",
            "X_test_feat_v1",
            "feature_set_v1_decision_log",
            "feature_set_version_label",
        ],
        "path_or_value": [
            str(X_train_feat_path.resolve()),
            str(X_test_feat_path.resolve()),
            str(feature_log_path.resolve()),
            FEATURE_SET_VERSION,
        ],
    }
)

feature_artifact_shape_summary = pd.DataFrame(
    {
        "dataset": ["X_train_feat_v1", "X_test_feat_v1"],
        "rows": [X_train_feat.shape[0], X_test_feat.shape[0]],
        "columns": [X_train_feat.shape[1], X_test_feat.shape[1]],
    }
)

display(feature_artifact_summary)
display(feature_artifact_shape_summary)


,artifact,path_or_value
0,X_train_feat_v1,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
1,X_test_feat_v1,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
2,feature_set_v1_decision_log,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
3,feature_set_version_label,feature_set_v1


,dataset,rows,columns
0,X_train_feat_v1,69577,44
1,X_test_feat_v1,17395,44


## Review Checklist

- [x] Every engineered feature has a documented source and rationale.
- [x] Every transformation is fit on training data only.
- [x] Final feature list is saved to `reports/tables/feature_set_v1.csv`.
- [x] No post-outcome information is used (leakage re-check).
- [x] Feature matrix shape reported; row order matches `y_train` / `y_test`.
- [x] High-cardinality encoding strategy documented (`country`, `agent`, `company`).
- [x] Feature-set version (`feature_set_v1`) is referenced correctly in downstream notebooks.


## Handoff To Notebook 05

- Frozen artifacts: `data/processed/X_train_feat_v1.csv`, `data/processed/X_test_feat_v1.csv`, and `reports/tables/feature_set_v1.csv`.
- Notebook 05 should use `feature_set_v1` unchanged for all baseline models so model comparisons stay tied to one fixed engineered feature set.
- Deferred feature decisions carried forward from this notebook: `room_mismatch` remains blocked by the `assigned_room_type` timing review, and raw top-k `agent`/`company` ID expansion remains postponed unless later model evidence justifies it.
- Any feature-set change during tuning in Notebook 06 should create a new version label such as `feature_set_v2` and be logged explicitly.
